# 🧠 Build a GPT-style LLM from Scratch (PyTorch only)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/loki52501/My_own_llm/blob/main/llm-from-scratch/LLM_From_Scratch.ipynb)

This notebook builds a small **GPT-style Large Language Model** end-to-end, using **only PyTorch** (plus the Python standard library, and matplotlib for plots).
Nothing is imported from `transformers`, `tiktoken`, `tokenizers`, or any other ML library. Every piece is written by hand.

It trains on **your own books**: the `txt-files.tar` archive of Project Gutenberg books in the Hugging Face dataset [`Lokeshlks/gutenberg_books_text`](https://huggingface.co/datasets/Lokeshlks/gutenberg_books_text). It streams just the part it needs.

### The roadmap

```
 ┌───────────┐   ┌────────────┐   ┌──────────────┐   ┌────────────┐   ┌───────────┐   ┌──────────┐   ┌────────────┐
 │ 1. Books  │──▶│ 2. BPE     │──▶│ 3. Dataset / │──▶│ 4. Embed-  │──▶│ 5. Self-  │──▶│ 6. GPT   │──▶│ 7. Train & │
 │  (.txt)   │   │ Tokenizer  │   │  DataLoader  │   │   dings    │   │ Attention │   │  Model   │   │  Generate  │
 └───────────┘   └────────────┘   └──────────────┘   └────────────┘   └───────────┘   └──────────┘   └────────────┘
   raw text       text → ids       (input, target)    ids → vectors   tokens "talk"   stack blocks    learn + write
```

| Step | What you build | Key idea |
|---|---|---|
| 1 | Load the books | An LLM only ever sees one long stream of text |
| 2 | Byte-Pair-Encoding tokenizer | Repeatedly merge the most frequent pair of symbols |
| 3 | Sliding-window dataset | Target = input shifted one token to the right |
| 4 | Token + positional embeddings | Every token id becomes a learnable vector, plus "where am I" information |
| 5 | Causal self-attention | Each token builds its meaning from a weighted mix of earlier tokens |
| 6 | Transformer block → GPT | Attention + MLP + LayerNorm + residuals, stacked N times |
| 7 | Training loop and sampling | Minimize next-token cross-entropy, then generate text one token at a time |

> **Colab tip:** `Runtime → Change runtime type → T4 GPU` before running. Training reads **every book** in the 30 GB dataset by streaming it. Each run trains for `SESSION_HOURS` (default 3 h), saves to Google Drive, and the next run continues where it stopped. It also runs on CPU, as a small demo.

This follows the same order as Sebastian Raschka's *Build a Large Language Model (From Scratch)* and Andrej Karpathy's *nanoGPT* / *minbpe*.

## 0. Setup

We only need `torch` (already installed on Colab) and `matplotlib` for plots. This cell imports everything, fixes the random seed, and picks the device.

In [ ]:
import os, re, io, sys, math, time, json, glob, heapq, tarfile, random, urllib.request, collections
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules

SEED = 1337
random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device = {device} | running in Colab = {IN_COLAB}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 1. The books: all of them, streamed 📚

The books come from the Hugging Face dataset **[`Lokeshlks/gutenberg_books_text`](https://huggingface.co/datasets/Lokeshlks/gutenberg_books_text)**: one `txt-files.tar` with **every** Project Gutenberg `.txt` book, **≈30 GB** in total.

We want the model to read **all of it**. 30 GB doesn't fit in Colab's RAM (≈12 GB), and one pass takes longer than one Colab session. So instead of loading the files, we **stream** them:

```
 Hugging Face txt-files.tar (30 GB)                         Google Drive (survives restarts)
 ┌────────┬────────┬────────┬────────┬─────── ··· ───┐       ┌──────────────────────────────────┐
 │ book 1 │ book 2 │ book 3 │ book 4 │  ...          │       │ tokenizer.json  sample.json      │
 └────────┴────────┴────────┴────────┴─────── ··· ───┘       │ checkpoint.pt:                   │
      ▲ HTTP "Range: bytes=<position>-"                      │   model + optimizer weights      │
      │ (start reading at any byte)                          │   step, stream position (bytes)  │
      │                                                      └──────────────────────────────────┘
 read one book ─▶ English? ─▶ strip licence ─▶ tokenize ─▶ windows ─▶ shuffle buffer ─▶ batches ─▶ GPU
                                                                            │
                        every few minutes: save model + byte position ──────┘  → next session resumes there
```

* A `.tar` is just files laid end to end, so Python's `tarfile` in **stream mode** (`"r|"`) reads it book by book, with no seeking and no full download.
* Every book passes a filter: we keep only `Language: English` books (optional) and cut the Project Gutenberg header and licence text.
* We remember the **byte position** of the next unread book. On restart, an HTTP `Range` request resumes the stream right there, so **over several sessions the model reads every book exactly once**.
* **1 book in 50** (picked by a hash of its file name) is never trained on: it's our **validation set**.

In this section we only read a **sample** (`SAMPLE_MB`) from the start of the stream. It's used to train the tokenizer, to build the validation set, and for the small demos in the next sections. The sample is saved, so every session uses the same tokenizer and the same validation books.


> **Tip:** the Hugging Face dataset viewer reads this tar as a *WebDataset*: one row per book, with the text in a `txt` column. With the `datasets` library, `load_dataset("Lokeshlks/gutenberg_books_text", split="train", streaming=True)` yields the same rows. Here we stream with the Python standard library, so there's nothing extra to install, and we can **resume at an exact byte position**.

In [ ]:
import zlib, http.client

HF_TAR_URL       = "https://huggingface.co/datasets/Lokeshlks/gutenberg_books_text/resolve/main/txt-files.tar"
DATASET_BYTES    = 30_245_222_400             # size of txt-files.tar
ENGLISH_ONLY     = True
VAL_EVERY        = 50                         # 1 in 50 books → validation, never trained on
SAMPLE_MB        = 30 if torch.cuda.is_available() else 4
USE_GOOGLE_DRIVE = IN_COLAB                   # keeps checkpoints + stream position between sessions

WORK_DIR = Path("llm_work"); WORK_DIR.mkdir(exist_ok=True)
if USE_GOOGLE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    SAVE_DIR = Path("/content/drive/MyDrive/llm_from_scratch")
else:
    SAVE_DIR = WORK_DIR
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print("saving to", SAVE_DIR)

EOT = "<|endoftext|>"

def clean(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("﻿", "")
    # keep only the book itself: drop the Project Gutenberg header and licence footer
    start = re.search(r"\*\*\* ?START OF (?:THE|THIS) PROJECT GUTENBERG.*?\*\*\*", text)
    end = re.search(r"\*\*\* ?END OF (?:THE|THIS) PROJECT GUTENBERG", text)
    if start:
        text = text[start.end():end.start() if end else None]
    return re.sub(r"\n{3,}", "\n\n", text).strip()

def is_val_book(name):
    return zlib.crc32(name.encode()) % VAL_EVERY == 0

def stream_books(start_offset=0, max_retries=50):
    """Yield (name, text, next_offset) for every kept book in the tar, starting at a byte offset.

    next_offset = where the *following* book starts, i.e. where to resume after this one.
    Network hiccups are handled by re-opening the stream at the last good position."""
    offset, retries = start_offset, 0
    while True:
        base = offset
        try:
            req = urllib.request.Request(HF_TAR_URL, headers={"Range": f"bytes={base}-"} if base else {})
            with urllib.request.urlopen(req, timeout=60) as src, tarfile.open(fileobj=src, mode="r|") as tar:
                for m in tar:
                    nxt = base + m.offset_data + (m.size + 511) // 512 * 512    # tar pads files to 512 bytes
                    if m.isfile() and m.name.lower().endswith(".txt"):
                        raw = tar.extractfile(m).read().decode("utf-8", errors="ignore")
                        if not ENGLISH_ONLY or re.search(r"^Language:\s*English", raw[:5000], re.M):
                            text = clean(raw)
                            if len(text) > 2_000:
                                yield m.name, text, nxt
                    offset = nxt
            return                                                      # reached the end of the tar 🎉
        except (OSError, EOFError, tarfile.TarError, http.client.HTTPException) as e:
            retries += 1
            if retries > max_retries:
                raise
            print(f"  stream interrupted ({e!r}); resuming at byte {offset:,}")
            time.sleep(min(60, 2 * retries))

sample_path = SAVE_DIR / f"sample_{SAMPLE_MB}MB.json"
if sample_path.exists():
    sample = json.loads(sample_path.read_text())
    print("loaded saved sample", sample_path)
else:
    sample, n, t0 = {"train": [], "val": []}, 0, time.time()
    for name, text, offset in stream_books(0):
        sample["val" if is_val_book(name) else "train"].append(text)
        n += len(text)
        if n >= SAMPLE_MB * 1e6:
            break
    sample["kept_chars"], sample["end_offset"] = n, offset     # to estimate the size of the full English corpus
    sample_path.write_text(json.dumps(sample))
    print(f"read {n/1e6:.1f} MB in {time.time()-t0:.0f}s")

raw_text = f"\n{EOT}\n".join(sample["train"])
val_text = f"\n{EOT}\n".join(sample["val"])
print(f"sample: {len(sample['train'])} train books ({len(raw_text)/1e6:.1f} MB) + "
      f"{len(sample['val'])} validation books ({len(val_text)/1e6:.1f} MB)")

In [ ]:
print("--- sample ---")
print(raw_text[20000:20600])

## 2. The tokenizer ✂️

Neural networks work on numbers, not strings. A **tokenizer** turns text into a list of integers (token ids) and back again.

```
 "The cat sat."  ──encode──▶  [464, 3797, 3332, 13]  ──decode──▶  "The cat sat."
```

### 2a. The simplest tokenizer: characters

Give every unique character an id. It's easy, but sequences get long (1 token per character), which wastes the model's limited context window.

In [ ]:
chars = sorted(set(raw_text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

char_encode = lambda s: [stoi[c] for c in s]
char_decode = lambda ids: "".join(itos[i] for i in ids)

s = "Hello, world!"
print(f"char vocab size = {len(chars)}")
print(s, "→", char_encode(s), "→", repr(char_decode(char_encode(s))))

### 2b. Byte-Pair Encoding (BPE), the way GPT does it

BPE starts from **raw bytes** (256 base tokens, so *any* text can be encoded, even emoji) and **learns merges**:

```
 Step 0 (bytes):   t h e _ c a t _ t h e _ h a t
 Most frequent pair: (t,h) → new token 256 "th"
 Step 1:           th e _ c a t _ th e _ h a t
 Most frequent pair: (th,e) → new token 257 "the"
 Step 2:           the _ c a t _ the _ h a t
 Most frequent pair: (a,t) → new token 258 "at"
 Step 3:           the _ c at _ the _ h at
 ...repeat until the vocabulary reaches VOCAB_SIZE
```

Frequent words end up as a single token and rare words get split into pieces, so nothing is ever "out of vocabulary".

**Pre-tokenization.** Like GPT-2, we first split the text into word-like chunks with a regex (`" cat"`, `","`, `"'s"`, ...). Merges never cross chunk boundaries. This keeps tokens sensible, and it makes training fast because we only need to count each *unique* chunk once, weighted by how often it appears.

**Speed trick.** Rather than recount every pair after each merge, we keep a running `pair → count` table and only update the words that contain the merged pair. A heap finds the most frequent pair quickly.

In [ ]:
# GPT-2-style pre-tokenizer, pure `re`:  contractions | words | 1-3 digit numbers | punctuation | whitespace
# ([^\W\d_] = any Unicode letter, so accented words like 'café' stay together)
PRETOKENIZE = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?[^\W\d_]+| ?\d{1,3}| ?[^\s\w]+| ?_+|\s+(?!\S)|\s+""")

print(PRETOKENIZE.findall("Hello there, I'm learning LLMs in 2024!!  Café au lait."))

In [ ]:
def merge_ids(ids, pair, new_id):
    """Replace every occurrence of `pair` in `ids` with `new_id`."""
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(new_id); i += 2
        else:
            out.append(ids[i]); i += 1
    return out


# <|endoftext|> separates books. The other six are reserved for the reasoning notebook
# (Reasoning_Model_From_Scratch.ipynb), which fine-tunes the model trained here.
SPECIAL_TOKENS = [EOT, "<|user|>", "<|assistant|>", "<think>", "</think>", "<answer>", "</answer>"]

class BPETokenizer:
    def __init__(self, special_tokens=SPECIAL_TOKENS):
        self.merges = {}                                   # (id_a, id_b) -> new_id   (order = priority)
        self.vocab = {i: bytes([i]) for i in range(256)}   # id -> bytes
        self.special_tokens = list(special_tokens)
        self.special = {}                                  # "<|endoftext|>" -> id
        self._cache = {}

    # ---------------------------------------------------------------- training
    def train(self, text, vocab_size, verbose_every=0):
        n_merges = vocab_size - 256 - len(self.special_tokens)
        assert n_merges >= 0

        # 1) count unique chunks (special tokens are removed from training text)
        for sp in self.special_tokens:
            text = text.replace(sp, " ")
        word_freq = collections.Counter(PRETOKENIZE.findall(text))
        words = [list(w.encode("utf-8")) for w in word_freq]     # each word as a list of byte ids
        freqs = list(word_freq.values())

        # 2) initial pair counts + index "which words contain this pair"
        pair_counts = collections.defaultdict(int)
        where = collections.defaultdict(set)
        for wi, (ids, f) in enumerate(zip(words, freqs)):
            for p in zip(ids, ids[1:]):
                pair_counts[p] += f
                where[p].add(wi)
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        # 3) merge loop
        for i in range(n_merges):
            # pop until the heap top matches the *current* count (lazy deletion)
            while heap:
                neg_c, pair = heapq.heappop(heap)
                if pair_counts.get(pair, 0) == -neg_c and -neg_c > 0:
                    break
            else:
                print("no more pairs to merge"); break

            new_id = 256 + i
            self.merges[pair] = new_id
            self.vocab[new_id] = self.vocab[pair[0]] + self.vocab[pair[1]]
            if verbose_every and (i < 10 or i % verbose_every == 0):
                print(f"merge {i+1:5d}: {self.vocab[pair[0]]!r:>12} + {self.vocab[pair[1]]!r:<12} -> {new_id} {self.vocab[new_id]!r}  (count {-neg_c:,})")

            # update only the words containing this pair
            changed = set()
            for wi in list(where[pair]):
                old = words[wi]
                new = merge_ids(old, pair, new_id)
                if new == old:
                    continue
                f = freqs[wi]
                for p in zip(old, old[1:]):
                    pair_counts[p] -= f
                    changed.add(p)
                for p in zip(new, new[1:]):
                    pair_counts[p] += f
                    where[p].add(wi)
                    changed.add(p)
                words[wi] = new
            for p in changed:
                if pair_counts[p] > 0:
                    heapq.heappush(heap, (-pair_counts[p], p))
            del pair_counts[pair], where[pair]

        # 4) special tokens get the last ids
        for sp in self.special_tokens:
            idx = len(self.vocab)
            self.special[sp] = idx
            self.vocab[idx] = sp.encode("utf-8")
        self._cache = {}
        return self

    @property
    def vocab_size(self):
        return len(self.vocab)

    # ---------------------------------------------------------------- encode / decode
    def _encode_chunk(self, chunk):
        if chunk in self._cache:
            return self._cache[chunk]
        if len(self._cache) > 500_000:          # streaming 30 GB: keep the cache from growing forever
            self._cache.clear()
        ids = list(chunk.encode("utf-8"))
        while len(ids) >= 2:
            # apply the merge that was learned *earliest* (lowest new id) first
            pair = min(zip(ids, ids[1:]), key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            ids = merge_ids(ids, pair, self.merges[pair])
        self._cache[chunk] = ids
        return ids

    def encode(self, text):
        ids = []
        # split out special tokens so they map to their own single id
        if self.special:
            pattern = "(" + "|".join(re.escape(s) for s in self.special) + ")"
            parts = re.split(pattern, text)
        else:
            parts = [text]
        for part in parts:
            if part in self.special:
                ids.append(self.special[part])
            elif part:
                for chunk in PRETOKENIZE.findall(part):
                    ids.extend(self._encode_chunk(chunk))
        return ids

    def decode(self, ids):
        return b"".join(self.vocab[i] for i in ids).decode("utf-8", errors="replace")

    # ---------------------------------------------------------------- save / load
    def save(self, path):
        json.dump({"merges": [[a, b, n] for (a, b), n in self.merges.items()],
                   "special": self.special}, open(path, "w"))

    @classmethod
    def load(cls, path):
        d = json.load(open(path))
        tok = cls(special_tokens=list(d["special"]))
        for a, b, n in d["merges"]:
            tok.merges[(a, b)] = n
            tok.vocab[n] = tok.vocab[a] + tok.vocab[b]
        for sp, i in d["special"].items():
            tok.special[sp] = i
            tok.vocab[i] = sp.encode("utf-8")
        return tok

### Train the tokenizer

`VOCAB_SIZE` is a trade-off. Bigger means fewer tokens per sentence (more text fits in the context), but a bigger embedding table and rarer tokens. GPT-2 uses 50,257. For a ~10M-parameter model, **8,192** is a good fit (4,096 on CPU).

The merges are learned from the **sample** (a few tens of MB is plenty to learn English sub-words). The tokenizer is saved to `SAVE_DIR`, and every later session **loads** it instead of retraining. That matters: a model only understands the exact token ids it was trained with.

In [ ]:
VOCAB_SIZE = 8192 if torch.cuda.is_available() else 4096
TOK_PATH = SAVE_DIR / "tokenizer.json"

if TOK_PATH.exists():
    tokenizer = BPETokenizer.load(TOK_PATH)
    print(f"loaded saved tokenizer ({tokenizer.vocab_size} tokens) from {TOK_PATH}")
else:
    t0 = time.time()
    tokenizer = BPETokenizer().train(raw_text, VOCAB_SIZE, verbose_every=1000)
    tokenizer.save(TOK_PATH)
    print(f"\nTrained BPE with vocab size {tokenizer.vocab_size} in {time.time()-t0:.1f}s → {TOK_PATH}")

### See the tokenizer in action

Each `|` below is a token boundary. Common words become one token, and rare words are split into sub-word pieces.

In [ ]:
sample = "The quick brown fox jumps over the lazy dog. Extraordinarily, tokenization works!"
ids = tokenizer.encode(sample)
pieces = [tokenizer.decode([i]) for i in ids]

print("text   :", sample)
print("ids    :", ids)
print("tokens :", "|".join(pieces))
print(f"\n{len(sample)} characters → {len(ids)} tokens   (a char-level tokenizer would need {len(sample)})")
assert tokenizer.decode(ids) == sample, "round-trip failed!"
print("round trip OK ✓")

print("\nSome learned tokens:", [tokenizer.vocab[i].decode('utf-8', 'replace') for i in range(1000, 1030)])

### Encode the sample

We turn the sample books (and the validation books) into tensors of token ids. The sample is used for the demos below and for the validation loss. The **real training data** is tokenized on the fly while streaming (section 3b).

In [ ]:
t0 = time.time()
train_data = torch.tensor(tokenizer.encode(raw_text), dtype=torch.long)
val_data = torch.tensor(tokenizer.encode(val_text), dtype=torch.long)
chars_per_token = len(raw_text) / len(train_data)
print(f"sample: {len(raw_text):,} chars → {len(train_data):,} tokens ({chars_per_token:.2f} chars/token) in {time.time()-t0:.1f}s")
print(f"validation: {len(val_data):,} tokens (from books the model never trains on)")

lengths = [len(tokenizer.vocab[i]) for i in range(256, tokenizer.vocab_size - len(tokenizer.special))]
plt.figure(figsize=(6, 3))
plt.hist(lengths, bins=range(1, 16)); plt.xlabel("bytes per learned token"); plt.ylabel("count")
plt.title("How long are the BPE tokens?"); plt.show()

## 3. Dataset: input → target pairs 🎯

A language model is trained to **predict the next token**. From one stream of ids we cut windows of length `context_length`. The **target is the input shifted by one**:

```
 token stream:  [ 40, 367, 2885, 1464, 1807, 3619, 402, 271, ... ]

                  ┌─────────── context_length = 4 ────────────┐
 input  x:      [ 40,   367,  2885, 1464 ]
 target y:      [ 367,  2885, 1464, 1807 ]      ← same window, one step to the right

 This single window gives 4 training examples at once:
     [40]                  → 367
     [40, 367]             → 2885
     [40, 367, 2885]       → 1464
     [40, 367, 2885, 1464] → 1807
```

The window slides along the text by `stride` tokens. If `stride == context_length` the windows don't overlap.

In [ ]:
class GPTDataset(torch.utils.data.Dataset):
    def __init__(self, ids, context_length, stride):
        self.ids, self.T = ids, context_length
        self.starts = range(0, len(ids) - context_length - 1, stride)

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, i):
        s = self.starts[i]
        chunk = self.ids[s : s + self.T + 1]
        return chunk[:-1], chunk[1:]          # x, y (y is x shifted by one)

def make_loader(ids, context_length, batch_size, stride=None, shuffle=True):
    ds = GPTDataset(ids, context_length, stride or context_length)
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=True)

# demo with a tiny context
demo_loader = make_loader(train_data, context_length=8, batch_size=2)
x, y = next(iter(demo_loader))
print("x:", x.shape, "\n", x)
print("y:", y.shape, "\n", y)
print("\nx[0] as text:", repr(tokenizer.decode(x[0].tolist())))
print("y[0] as text:", repr(tokenizer.decode(y[0].tolist())))

### 3b. Streaming **every** book: `IterableDataset` + shuffle buffer

The `GPTDataset` above needs all the tokens in memory. For the full 30 GB we use a PyTorch **`IterableDataset`** instead. It *produces* training windows on the fly, while it reads the tar:

```
 stream_books() ─▶ tokenize book ─▶ append to a token "carry" ─▶ cut (context_length+1) windows ─┐
      ▲                                                                                              ▼
      │                                                       ┌───────── shuffle buffer (N windows) ─────────┐
  byte position                                               │ w17  w3  w250  w9  ...  (from many books)    │
  (saved in checkpoints)                                      └─── pick a random one ──▶ (x, y) ──▶ batch ───┘
```

The **shuffle buffer** matters. Books arrive one at a time, so without it every batch would come from the same book, and the model would keep "switching topics". Taking random windows from a buffer of tens of thousands mixes many books into each batch.

`dataset.position` always holds the byte offset of the next unread book, and we save it in every checkpoint. (On resume, the few windows still sitting in the buffer are skipped. That's a tiny fraction of the data.)

In [ ]:
class StreamingTokenDataset(torch.utils.data.IterableDataset):
    """Streams books → token ids → (x, y) windows, mixed through a shuffle buffer."""

    def __init__(self, tokenizer, context_length, start_offset=0, buffer_size=20_000, seed=0):
        self.tok, self.T = tokenizer, context_length
        self.start_offset, self.position = start_offset, start_offset
        self.buffer_size, self.seed = buffer_size, seed
        self.books_read, self.finished = 0, False

    def __iter__(self):
        rng, buf, carry = random.Random(self.seed + self.start_offset), [], []
        eot = self.tok.special[EOT]
        for name, text, nxt in stream_books(self.start_offset):
            self.position = nxt
            if is_val_book(name):                       # validation books are never trained on
                continue
            carry.extend(self.tok.encode(text) + [eot])
            i = 0
            while i + self.T + 1 <= len(carry):         # cut non-overlapping windows of T+1 tokens
                buf.append(torch.tensor(carry[i:i + self.T + 1]))
                i += self.T
            carry = carry[i:]
            self.books_read += 1
            while len(buf) > self.buffer_size:          # emit random windows once the buffer is full
                j = rng.randrange(len(buf))
                buf[j], buf[-1] = buf[-1], buf[j]
                w = buf.pop()
                yield w[:-1], w[1:]
        self.finished = True                            # the whole tar has been read
        rng.shuffle(buf)
        for w in buf:
            yield w[:-1], w[1:]

In [ ]:
# peek at the live stream: 4 windows of 24 tokens, from a small shuffle buffer
peek = StreamingTokenDataset(tokenizer, context_length=24, buffer_size=200)
xb, yb = next(iter(torch.utils.data.DataLoader(peek, batch_size=4)))
for row in xb:
    print(repr(tokenizer.decode(row.tolist())))
print(f"\nstream position: byte {peek.position:,} of {DATASET_BYTES:,}")

## 4. Embeddings: from ids to vectors 🔢➡️📐

A token id like `2885` has no meaning by itself. An **embedding layer** is a lookup table with one learnable row (vector) per token:

```
   token embedding table (vocab_size × d_model)       positional embedding table (context_length × d_model)
          ┌──────────────────────────────┐                   ┌──────────────────────────────┐
     0    │ 0.12 -0.40  0.88 ...  0.05   │              0    │ ...                          │
     1    │ ...                          │              1    │ ...                          │
     ⋮    │                              │              ⋮    │                              │
  2885 ──▶│ 0.31  0.07 -1.20 ...  0.66   │         pos p ──▶ │ 0.02 -0.11  0.45 ... -0.30   │
     ⋮    │                              │              ⋮    │                              │
          └──────────────────────────────┘                   └──────────────────────────────┘

   x = token_embedding[id]  +  position_embedding[position]        → shape (batch, T, d_model)
```

Attention by itself has **no sense of order**: "dog bites man" and "man bites dog" would look identical. So we **add** a learned position vector. During training, similar tokens end up with similar vectors.

In [ ]:
d_demo = 16
tok_emb = nn.Embedding(tokenizer.vocab_size, d_demo)
pos_emb = nn.Embedding(8, d_demo)

xb, _ = next(iter(demo_loader))                     # (B=2, T=8) ids
te = tok_emb(xb)                                    # (2, 8, 16)
pe = pos_emb(torch.arange(xb.shape[1]))             # (8, 16), broadcast over batch
h = te + pe
print("ids", tuple(xb.shape), "→ token emb", tuple(te.shape), "+ pos emb", tuple(pe.shape), "→", tuple(h.shape))

## 5. Self-attention, step by step 👀

This is the heart of the Transformer. Each token asks *"which earlier tokens are relevant to me?"* and builds a new vector as a **weighted average** of them.

Each token vector `x` is projected into three vectors:

| | Meaning | Analogy |
|---|---|---|
| **Query** `q = x·W_q` | what I'm looking for | a search query |
| **Key** `k = x·W_k` | what I contain | the title of a document |
| **Value** `v = x·W_v` | what I'll hand over if chosen | the document's content |

```
          ┌─────┐
     ┌───▶│ W_q │──▶ Q ──┐
     │    └─────┘        │
     │    ┌─────┐        │     ┌──────────────┐    ┌─────────────┐    ┌─────────┐
  X ─┼───▶│ W_k │──▶ K ──┴───▶ │ Q·Kᵀ / √d    │──▶ │ causal mask │──▶ │ softmax │──▶ A (T×T)
     │    └─────┘              │ scores (T×T) │    │ future=-inf │    └─────────┘      │
     │    ┌─────┐              └──────────────┘    └─────────────┘                     ▼
     └───▶│ W_v │──▶ V ─────────────────────────────────────────────────────────▶  out = A · V
          └─────┘
```

**Causal mask.** When predicting token *t* the model must not peek at tokens *t+1, t+2, ...*, so we set those scores to `-inf` before the softmax (and they become 0):

```
            keys →   The  cat  sat  on
 queries ↓  The    [  ✓    ✗    ✗    ✗ ]
            cat    [  ✓    ✓    ✗    ✗ ]
            sat    [  ✓    ✓    ✓    ✗ ]
            on     [  ✓    ✓    ✓    ✓ ]
```

Let's do it by hand on a tiny example.

In [ ]:
torch.manual_seed(0)
words = ["The", " cat", " sat", " on", " the", " mat"]
T, d = len(words), 8
X = torch.randn(T, d)                        # pretend these are embeddings for 6 tokens

W_q, W_k, W_v = (torch.randn(d, d) for _ in range(3))
Q, K, V = X @ W_q, X @ W_k, X @ W_v          # (T, d) each

# 1) similarity of every query with every key
scores = Q @ K.T / math.sqrt(d)              # (T, T), scaled so softmax isn't too peaky
# 2) causal mask: no looking into the future
mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
scores = scores.masked_fill(mask, float("-inf"))
# 3) softmax → each row sums to 1
A = torch.softmax(scores, dim=-1)
# 4) weighted sum of values
out = A @ V                                   # (T, d)

print("row sums:", A.sum(-1))
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(A.detach(), cmap="viridis")
ax.set_xticks(range(T), words, rotation=45); ax.set_yticks(range(T), words)
ax.set_xlabel("key (attended to)"); ax.set_ylabel("query (attending)")
ax.set_title("Causal attention weights (random, untrained)")
plt.colorbar(im); plt.show()

### Multi-head attention

One attention "head" can focus on only one kind of relationship at a time. **Multi-head attention** runs `n_heads` smaller attentions in parallel, for example one head tracking grammar and another tracking which name "she" refers to. It then concatenates the results and mixes them with an output projection.

```
 X ─▶ Linear(d, 3d) ─▶ split into Q, K, V ─▶ reshape to (B, n_heads, T, head_dim)
                                                │  head 1: softmax(QKᵀ/√hd + mask)·V
                                                │  head 2: ...
                                                │  head h: ...
                                                ▼
                     concat heads (B, T, d) ─▶ Linear(d, d) ─▶ dropout ─▶ out
```

We implement the math manually. `F.scaled_dot_product_attention` (also plain PyTorch) computes exactly the same thing with a fused, faster kernel, and we switch to it when `use_fused=True`.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, context_length, dropout=0.0, use_fused=True):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)   # W_q, W_k, W_v in one matmul
        self.proj = nn.Linear(d_model, d_model)
        self.attn_drop, self.resid_drop = nn.Dropout(dropout), nn.Dropout(dropout)
        self.dropout, self.use_fused = dropout, use_fused
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), 1),
                             persistent=False)

    def forward(self, x, return_attn=False):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # (B, T, C) -> (B, n_heads, T, head_dim)
        q, k, v = (t.view(B, T, self.n_heads, self.head_dim).transpose(1, 2) for t in (q, k, v))

        if self.use_fused and not return_attn:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                               dropout_p=self.dropout if self.training else 0.0)
            att = None
        else:
            att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)   # (B, h, T, T)
            att = att.masked_fill(self.mask[:T, :T], float("-inf"))
            att = torch.softmax(att, dim=-1)
            y = self.attn_drop(att) @ v                                  # (B, h, T, head_dim)

        y = y.transpose(1, 2).contiguous().view(B, T, C)                 # concat heads
        y = self.resid_drop(self.proj(y))
        return (y, att) if return_attn else y

mha = CausalSelfAttention(d_model=16, n_heads=4, context_length=8)
print("in", tuple(h.shape), "→ out", tuple(mha(h).shape))

# sanity check: manual math == fused kernel
mha.eval()
y_manual, _ = mha(h, return_attn=True)
y_fused = mha(h)
print("manual vs fused max diff:", (y_manual - y_fused).abs().max().item())

## 6. The Transformer block and the full GPT 🏗️

### Building blocks

* **LayerNorm** rescales each token vector to mean 0 and variance 1 (plus a learnable scale and shift). This keeps training stable.
* **Feed-forward (MLP)**: `Linear(d → 4d) → GELU → Linear(4d → d)`, applied to each token independently. Attention *mixes information between tokens*; the MLP *processes* it within each token.
* **Residual connections** (`x = x + f(x)`) give gradients a "highway" through deep networks.

### One Transformer block (pre-LayerNorm, as in GPT-2)

```
        x ─────────────────────────┐                       x ───────────────────────┐
        │                          │                       │                        │
        ▼                          │                       ▼                        │
   ┌──────────┐   ┌────────────┐   │   (+)            ┌──────────┐   ┌─────────┐    │   (+)
   │LayerNorm │──▶│ Multi-Head │───┴──▶ ● ──────────▶ │LayerNorm │──▶│   MLP   │────┴──▶ ● ──▶ out
   └──────────┘   │ Attention  │                      └──────────┘   │(4× GELU)│
                  └────────────┘                                     └─────────┘
```

### The whole GPT

```
  token ids (B, T)
        │
        ▼
  ┌────────────────────────┐  ┌──────────────────────────┐
  │ Token embedding (V×d)  │ +│ Position embedding (T×d) │
  └────────────────────────┘  └──────────────────────────┘
        │  dropout
        ▼
  ┌────────────────────────┐
  │  Transformer block 1   │
  │  Transformer block 2   │   × n_layers
  │          ...           │
  │  Transformer block N   │
  └────────────────────────┘
        │
        ▼
    Final LayerNorm
        │
        ▼
  Linear head (d → V)   ← shares weights with the token embedding ("weight tying")
        │
        ▼
  logits (B, T, V)  →  softmax  →  probability of every possible next token
```

In [ ]:
class LayerNorm(nn.Module):
    """LayerNorm written out by hand (same as nn.LayerNorm)."""
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))
        self.bias = nn.Parameter(torch.zeros(d))

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)
        return self.weight * (x - mean) / torch.sqrt(var + self.eps) + self.bias


class GELU(nn.Module):
    """Gaussian Error Linear Unit (tanh approximation used by GPT-2)."""
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * x ** 3)))


class FeedForward(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, 4 * d_model), GELU(),
                                 nn.Linear(4 * d_model, d_model), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = LayerNorm(cfg["d_model"])
        self.attn = CausalSelfAttention(cfg["d_model"], cfg["n_heads"], cfg["context_length"],
                                        cfg["dropout"], cfg.get("use_fused", True))
        self.ln2 = LayerNorm(cfg["d_model"])
        self.ff = FeedForward(cfg["d_model"], cfg["dropout"])

    def forward(self, x):
        x = x + self.attn(self.ln1(x))     # residual 1
        x = x + self.ff(self.ln2(x))       # residual 2
        return x


# visualize GELU vs ReLU
xs = torch.linspace(-4, 4, 200)
plt.figure(figsize=(5, 3))
plt.plot(xs, GELU()(xs), label="GELU"); plt.plot(xs, F.relu(xs), "--", label="ReLU")
plt.legend(); plt.grid(alpha=.3); plt.title("GELU is a smooth ReLU"); plt.show()

In [ ]:
class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["d_model"])
        self.drop = nn.Dropout(cfg["dropout"])
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.ln_f = LayerNorm(cfg["d_model"])
        self.head = nn.Linear(cfg["d_model"], cfg["vocab_size"], bias=False)
        self.head.weight = self.tok_emb.weight          # weight tying (saves V×d parameters)

        self.apply(self._init_weights)
        # GPT-2 trick: scale down the last projection of each residual branch
        for name, p in self.named_parameters():
            if name.endswith("proj.weight") or name.endswith("net.2.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg["n_layers"]))

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg["context_length"], "sequence longer than context_length"
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))     # (B, T, d)
        for block in self.blocks:
            x = block(x)
        logits = self.head(self.ln_f(x))                         # (B, T, vocab)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

## 7. Pick a model size and a training plan ⚙️

| Hyperparameter | Meaning |
|---|---|
| `context_length` | how many tokens the model can "see" at once |
| `d_model` | size of every token vector |
| `n_heads` | number of attention heads (must divide `d_model`) |
| `n_layers` | number of stacked Transformer blocks |
| `dropout` | randomly zeroes activations during training. We have so much data that we never repeat a book, so it's set to 0 |

**The training plan.** `TOTAL_STEPS` is set so that one pass over the **whole English part of the dataset** takes exactly `TOTAL_STEPS` batches. We estimate it from the sample (the English fraction and chars per token). The learning-rate schedule is laid out over that whole pass.

`SESSION_HOURS` is how long *this* run trains before it saves and stops. **Re-run the notebook to continue**: it resumes from the checkpoint and the exact stream position in Google Drive. On a T4, the default model processes roughly 100–150k tokens/s, so a full pass over ≈8B tokens takes on the order of **15–20 GPU-hours**, spread over as many sessions as you like.

On CPU the plan shrinks to a 300-step demo.

In [ ]:
if device == "cuda":
    cfg = dict(vocab_size=tokenizer.vocab_size, context_length=256, d_model=384, n_heads=6, n_layers=6, dropout=0.0)
    B, LR, WARMUP = 64, 6e-4, 1000
    SESSION_HOURS, EVAL_EVERY, CKPT_MINUTES = 3.0, 500, 15
else:   # CPU / Apple MPS: a toy model so it still runs
    cfg = dict(vocab_size=tokenizer.vocab_size, context_length=128, d_model=128, n_heads=4, n_layers=4, dropout=0.0)
    B, LR, WARMUP = 16, 1e-3, 30
    SESSION_HOURS, EVAL_EVERY, CKPT_MINUTES = 1.0, 50, 10

tokens_per_step = B * cfg["context_length"]
english_fraction = sample["kept_chars"] / sample["end_offset"]          # kept English chars per byte of tar
est_tokens = DATASET_BYTES * english_fraction / chars_per_token
TOTAL_STEPS = int(est_tokens / tokens_per_step) if device == "cuda" else 300
print(f"estimated English corpus: ≈{est_tokens/1e9:.1f}B tokens → one full pass = {TOTAL_STEPS:,} steps of {tokens_per_step:,} tokens")
print(f"this session trains for up to {SESSION_HOURS} h, then saves; re-run to continue")

torch.manual_seed(SEED)
model = GPT(cfg).to(device)
print(json.dumps(cfg, indent=1))
print(f"\nParameters: {model.num_params()/1e6:.2f} M")
for name, mod in [("embeddings", model.tok_emb), ("pos emb", model.pos_emb), ("one block", model.blocks[0])]:
    print(f"  {name:<11} {sum(p.numel() for p in mod.parameters()):>10,}")

### Before training: the model is random

The expected loss for random guessing is `ln(vocab_size)`. Let's check we're close to it, and look at what an untrained model "writes".

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0, top_k=None):
    """Autoregressive sampling: predict → sample → append → repeat."""
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.cfg["context_length"]:]         # crop to the context window
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / max(temperature, 1e-6)       # only the last position matters
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")          # keep only the top-k candidates
        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    return idx

def complete(prompt, max_new_tokens=80, temperature=0.8, top_k=50):
    idx = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)
    return tokenizer.decode(generate(model, idx, max_new_tokens, temperature, top_k)[0].tolist())

xb, yb = next(iter(make_loader(train_data, cfg["context_length"], 4)))
_, loss = model(xb.to(device), yb.to(device))
print(f"initial loss {loss.item():.3f}   vs   ln(vocab) = {math.log(cfg['vocab_size']):.3f}")
print("\nUntrained sample:\n", complete("It was a dark and stormy night", 40))

## 8. Training 🏋️

The training loop is the same for every neural network:

```
   ┌──────────────────────────────────────────────────────────────────────┐
   │ 1. get batch (x, y)                                                  │
   │ 2. forward:   logits = model(x)                                      │
   │ 3. loss:      cross_entropy(logits, y)   "how surprised were we?"    │
   │ 4. backward:  loss.backward()            gradients for all weights   │
   │ 5. step:      optimizer.step()           nudge weights downhill      │
   └────────────────────────────── repeat ────────────────────────────────┘
```

Extras used by real LLM training:

* **AdamW** optimizer with weight decay (not applied to biases or LayerNorm weights),
* **Learning-rate schedule**: linear warm-up, then cosine decay,
* **Gradient clipping** at 1.0 to avoid rare explosive updates,
* **Mixed precision** (`torch.autocast`) on GPU for about 2× speed,
* **Checkpointing and resuming**: model, optimizer, step, and **stream byte position** are saved to Google Drive every `CKPT_MINUTES`.

```
 session 1:  step 0 ───────────▶ step 40k  [save: weights + optimizer + byte 3.1 GB]
 session 2:                      step 40k ─────────▶ step 80k  [save: ... byte 6.3 GB]
   ...                                                                ...
 session N:                                                  ... ───▶ end of tar 🎉  every book read once
```

In [ ]:
def lr_at(step, max_lr, warmup, max_steps, min_ratio=0.1):
    if step < warmup:
        return max_lr * (step + 1) / warmup
    progress = (step - warmup) / max(1, max_steps - warmup)
    return max_lr * (min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * min(progress, 1.0))))

steps = range(0, TOTAL_STEPS, max(1, TOTAL_STEPS // 1000))
plt.figure(figsize=(6, 2.5))
plt.plot(steps, [lr_at(s, LR, WARMUP, TOTAL_STEPS) for s in steps])
plt.title("Learning-rate schedule over one full pass"); plt.xlabel("step"); plt.ylabel("lr"); plt.grid(alpha=.3)
plt.show()

In [ ]:
# --- optimizer: decay only 2D weight matrices ---
decay     = [p for n, p in model.named_parameters() if p.dim() >= 2]
no_decay  = [p for n, p in model.named_parameters() if p.dim() < 2]
optimizer = torch.optim.AdamW([{"params": decay, "weight_decay": 0.1},
                               {"params": no_decay, "weight_decay": 0.0}], lr=LR, betas=(0.9, 0.95))

# --- mixed precision ---
use_amp   = device == "cuda"
amp_dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16
scaler    = torch.amp.GradScaler("cuda", enabled=use_amp and amp_dtype == torch.float16)

# --- fixed validation windows (same ones every session, so the curve is comparable) ---
T = cfg["context_length"]
EVAL_WINDOWS = 50 * B if device == "cuda" else 10 * B
val_starts = torch.randint(len(val_data) - T - 1, (EVAL_WINDOWS,), generator=torch.Generator().manual_seed(0))

@torch.no_grad()
def estimate_val_loss():
    model.eval()
    losses = []
    for i in range(0, EVAL_WINDOWS, B):
        idx = val_starts[i:i + B]
        x = torch.stack([val_data[j:j + T] for j in idx]).to(device)
        y = torch.stack([val_data[j + 1:j + T + 1] for j in idx]).to(device)
        with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp):
            losses.append(model(x, y)[1].item())
    model.train()
    return sum(losses) / len(losses)

# --- resume from Google Drive if a checkpoint exists ---
CKPT_PATH = SAVE_DIR / "gpt_from_scratch.pt"
step, start_offset = 0, 0
history = {"step": [], "train": [], "val": [], "position": []}
if CKPT_PATH.exists():
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
    if ckpt["cfg"] == cfg:
        model.load_state_dict(ckpt["model"]); optimizer.load_state_dict(ckpt["optimizer"]); scaler.load_state_dict(ckpt["scaler"])
        step, start_offset, history = ckpt["step"], ckpt["position"], ckpt["history"]
        print(f"▶ resuming at step {step:,}, stream byte {start_offset:,} ({start_offset/DATASET_BYTES:.1%} of the dataset)")
    else:
        CKPT_PATH.rename(CKPT_PATH.with_suffix(".old.pt"))
        print("config changed: old checkpoint renamed to *.old.pt, starting fresh")

def save_checkpoint(position):
    tmp = CKPT_PATH.with_suffix(".tmp")
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
                "cfg": cfg, "step": step, "position": position, "history": history}, tmp)
    tmp.replace(CKPT_PATH)                      # atomic: never leaves a half-written checkpoint

In [ ]:
dataset = StreamingTokenDataset(tokenizer, T, start_offset=start_offset)
loader = torch.utils.data.DataLoader(dataset, batch_size=B)      # num_workers=0 → dataset.position stays live
PROMPT = "The"
t0 = last_ckpt = time.time()
step0, recent = step, []
model.train()

for x, y in loader:
    if step >= TOTAL_STEPS or time.time() - t0 > SESSION_HOURS * 3600:
        break

    # ---- one optimization step ----
    for g in optimizer.param_groups:
        g["lr"] = lr_at(step, LR, WARMUP, TOTAL_STEPS)
    x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
    with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp):
        _, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    recent.append(loss.item())
    step += 1

    # ---- evaluate every so often ----
    if step % EVAL_EVERY == 0:
        val = estimate_val_loss()
        history["step"].append(step); history["train"].append(sum(recent) / len(recent))
        history["val"].append(val); history["position"].append(dataset.position)
        recent = []
        tok_s = (step - step0) * tokens_per_step / (time.time() - t0)
        print(f"step {step:7,}/{TOTAL_STEPS:,} | train {history['train'][-1]:.3f} | val {val:.3f} | "
              f"read {dataset.position/DATASET_BYTES:6.2%} of dataset | {tok_s/1e3:5.0f}k tok/s | {(time.time()-t0)/60:5.1f} min")
        print("   ↳", complete(PROMPT, 30).replace("\n", " ")[:160])
        model.train()

    if time.time() - last_ckpt > CKPT_MINUTES * 60:
        save_checkpoint(dataset.position); last_ckpt = time.time()

save_checkpoint(dataset.position)
if dataset.finished:
    print("🎉 Every book in the dataset has been read.")
else:
    print(f"Session done: {step:,} steps total, {dataset.position/DATASET_BYTES:.2%} of the dataset read "
          f"({dataset.books_read} books this session). Re-run the notebook to continue.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["step"], history["train"], label="train (running avg)")
ax1.plot(history["step"], history["val"], label="validation")
ax1.set_xlabel("step"); ax1.set_ylabel("cross-entropy loss"); ax1.set_title("Training progress (all sessions)")
ax1.legend(); ax1.grid(alpha=.3)
ax2.plot(history["step"], [p / DATASET_BYTES * 100 for p in history["position"]])
ax2.set_xlabel("step"); ax2.set_ylabel("% of dataset read"); ax2.set_title("How far through the 30 GB of books")
ax2.grid(alpha=.3); plt.show()

**Reading the curve.** Both losses should fall together. Because the model never sees the same book twice, it can't memorize, so the train and validation losses stay close. The gap is **overfitting**, and it only opens up if you train on the same text over and over.

Perplexity `= e^loss` is roughly "how many tokens the model is choosing between". Lower is better.

## 9. Generate text ✍️

Generation is a loop: feed the prompt, take the probabilities for the **last** position, **sample** a token, append it, and repeat.

```
 "Once upon a"  ──model──▶ P(next) = { " time": 0.62, " day": 0.08, " hill": 0.03, ... } ──sample──▶ " time"
 "Once upon a time" ──model──▶ ...
```

Two knobs:

* **temperature**: `<1` is safer and more repetitive, `>1` is more creative and more chaotic,
* **top-k**: sample only from the `k` most likely tokens.

In [ ]:
torch.manual_seed(42)
for temp in [0.5, 0.8, 1.2]:
    print(f"━━━ temperature {temp} ━━━")
    print(complete("Once upon a time", max_new_tokens=100, temperature=temp, top_k=50))
    print()

In [ ]:
# ✏️ Try your own prompt!
prompt = "She opened the door and"
print(complete(prompt, max_new_tokens=150, temperature=0.8, top_k=40))

### Peek inside: what does the trained model attend to?

We run one sentence through the first block and plot each head's attention weights. After training, heads usually specialize: some look at the previous token, some at the start of the sentence, some at matching words.

In [ ]:
sentence = "The old man looked at the sea and smiled."
ids = torch.tensor([tokenizer.encode(sentence)], device=device)
labels = [tokenizer.decode([i]) for i in ids[0].tolist()]

model.eval()
with torch.no_grad():
    x = model.tok_emb(ids) + model.pos_emb(torch.arange(ids.shape[1], device=device))
    _, att = model.blocks[0].attn(model.blocks[0].ln1(x), return_attn=True)   # (1, heads, T, T)

n_show = min(4, cfg["n_heads"])
fig, axes = plt.subplots(1, n_show, figsize=(4.2 * n_show, 4))
for h_i, ax in enumerate(axes):
    ax.imshow(att[0, h_i].float().cpu(), cmap="viridis")
    ax.set_xticks(range(len(labels)), labels, rotation=90, fontsize=8)
    ax.set_yticks(range(len(labels)), labels, fontsize=8)
    ax.set_title(f"layer 1 · head {h_i+1}")
plt.tight_layout(); plt.show()

### Which tokens ended up with similar embeddings?

Similar words should now have similar embedding vectors. Here are the nearest neighbours by cosine similarity:

In [ ]:
E = F.normalize(model.tok_emb.weight.detach().float(), dim=-1)
for word in [" king", " love", " night", " said", " house"]:
    ids_w = tokenizer.encode(word)
    if len(ids_w) != 1:
        print(f"{word!r} is split into {len(ids_w)} tokens, skipping"); continue
    sims = E @ E[ids_w[0]]
    best = sims.topk(8).indices.tolist()[1:]
    print(f"{word!r:>10} → {[tokenizer.decode([b]) for b in best]}")

## 10. The saved model 💾

Everything is already in `SAVE_DIR` (Google Drive in Colab):

| File | What |
|---|---|
| `tokenizer.json` | the BPE merges + special tokens |
| `sample_*MB.json` | the sample books (tokenizer + validation) |
| `gpt_from_scratch.pt` | weights, optimizer, step, **stream position**, loss history |

**Reasoning_Model_From_Scratch.ipynb** loads `tokenizer.json` + `gpt_from_scratch.pt` from the same folder and turns this model into a reasoning model. Let's check that the checkpoint reloads cleanly:

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model2 = GPT(ckpt["cfg"]).to(device)
model2.load_state_dict(ckpt["model"])
tok2 = BPETokenizer.load(TOK_PATH)
assert tok2.encode("Hello world") == tokenizer.encode("Hello world")
print(f"reloaded model ({CKPT_PATH.stat().st_size/1e6:.0f} MB) + tokenizer ✓  | step {ckpt['step']:,} | "
      f"{ckpt['position']/DATASET_BYTES:.2%} of the dataset read")

## 🎉 Recap and next steps

You built every part of a GPT by hand:

```
 books ─▶ BPE tokenizer ─▶ sliding windows ─▶ embeddings ─▶ [ attention + MLP ] × N ─▶ logits ─▶ next token
```

**Ideas to go further**

1. **Scale up**: more books, `d_model=768, n_layers=12, n_heads=12` (GPT-2 small shape), and more steps.
2. **Better tokenizer**: a bigger `VOCAB_SIZE` (8k–32k) once you have lots of text.
3. **Modern tricks**: RoPE positional encoding, RMSNorm, SwiGLU MLPs, and a KV-cache for fast generation (the Llama recipe).
4. **Fine-tuning**: continue training on a smaller, specific text (one author, or dialogue) at a lower learning rate.
5. **Instruction tuning**: train on `### Instruction / ### Response` pairs to make a small chat model.

**References:** Vaswani et al., *Attention Is All You Need* (2017) · Radford et al., *GPT-2* (2019) · Sebastian Raschka, *Build a Large Language Model (From Scratch)* · Andrej Karpathy, *nanoGPT* and *minbpe*.